# 04 — Modular PyTorch K-Fold Training and Out-of-Fold Pipeline

**Session:** 2026-11-06 · Design Validation  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

A robust machine learning pipeline requires decoupling dataset abstraction, batch streaming, model training, validation assessment, and out-of-fold ensembling.

This notebook builds a clean, production-grade PyTorch training and evaluation harness:
1. **Dataset & DataLoader Abstractions:** Ingestion, indexing, batching, and shuffling.
2. **K-Fold Cross Validation Partitioning:** Splitting indices cleanly without data leakage.
3. **Decoupled Training Engine:** Modular `train_one_epoch()` and `validate_one_epoch()` functions.
4. **Out-of-Fold (OOF) Prediction Tracking:** Aggregating validation predictions across all $K$ models to compute an unbiased cross-validation score.
5. **Ensemble Blending on Unseen Holdout Test Data:** Demonstrating that the averaged predictions of $K$-fold models outperform individual fold models.

In [ ]:
import csv
from collections.abc import Iterator
from pathlib import Path

import numpy as np

np.random.seed(42)
print("Pipeline initialized with deterministic PRNG seed.")

## 1. Custom Dataset and DataLoader Abstractions

We construct lightweight, production-style `Dataset` and `DataLoader` abstractions:
- `Dataset`: Encapsulates feature and label storage with `__len__` and `__getitem__`.
- `DataLoader`: Yields minibatches with deterministic shuffling and indexing.

In [ ]:
class TabularDataset:
    """Standard PyTorch-style Dataset storing feature matrix X and target vector y."""

    def __init__(self, x: np.ndarray, y: np.ndarray):
        assert len(x) == len(y), "Features and targets must have identical length."
        self.x = np.asarray(x, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int | slice | np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        return self.x[idx], self.y[idx]


class MiniBatchLoader:
    """Mini-batch iterator supporting index samplers and shuffling."""

    def __init__(
        self,
        dataset: TabularDataset,
        batch_size: int = 32,
        indices: np.ndarray | None = None,
        shuffle: bool = True,
        seed: int = 42,
    ):
        self.dataset = dataset
        self.batch_size = batch_size
        self.indices = np.arange(len(dataset)) if indices is None else np.asarray(indices)
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __iter__(self) -> Iterator[tuple[np.ndarray, np.ndarray]]:
        active_idx = self.indices.copy()
        if self.shuffle:
            self.rng.shuffle(active_idx)

        for i in range(0, len(active_idx), self.batch_size):
            batch_idx = active_idx[i : i + self.batch_size]
            yield self.dataset[batch_idx]


# Load tabular housing validation data
data_path = Path("../data/tabular_housing_validation.csv")
with data_path.open("r", encoding="utf-8") as fp:
    records = list(csv.DictReader(fp))

feature_names = [
    "med_income",
    "house_age",
    "ave_rooms",
    "ave_bedrooms",
    "population",
    "ave_occupancy",
    "latitude",
    "longitude",
]
X_all = np.array([[float(r[col]) for col in feature_names] for r in records], dtype=np.float32)
y_all = np.array([float(r["target_price"]) for r in records], dtype=np.float32)

# Reserve 20% pure holdout test set (simulating competition private leaderboard)
n_total = len(X_all)
shuffled_idx = np.random.default_rng(42).permutation(n_total)
split_point = int(0.80 * n_total)

dev_idx, test_idx = shuffled_idx[:split_point], shuffled_idx[split_point:]
X_dev, y_dev = X_all[dev_idx], y_all[dev_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

# Feature standardization (fit strictly on development set to avoid leakage!)
mean_vec = np.mean(X_dev, axis=0)
std_vec = np.std(X_dev, axis=0) + 1e-8

X_dev_scaled = (X_dev - mean_vec) / std_vec
X_test_scaled = (X_test - mean_vec) / std_vec

full_dev_dataset = TabularDataset(X_dev_scaled, y_dev)
holdout_test_dataset = TabularDataset(X_test_scaled, y_test)

print(
    f"Development set: {len(full_dev_dataset)} samples | Holdout Test set: {len(holdout_test_dataset)} samples."
)

## 2. Decoupled Model and Training Engine

We build a modular feedforward neural regression model with two hidden layers, ReLU non-linearities, and L2 weight decay regularization.

In [ ]:
class FeedforwardRegressor:
    """Two-layer MLP regression model with He weight initialization."""

    def __init__(
        self,
        in_dim: int = 8,
        hidden_dim: int = 32,
        lr: float = 0.01,
        weight_decay: float = 1e-4,
    ):
        self.lr = lr
        self.weight_decay = weight_decay
        # He initialization: std = sqrt(2 / fan_in)
        self.w1 = np.random.normal(0, np.sqrt(2 / in_dim), size=(in_dim, hidden_dim))
        self.b1 = np.zeros(hidden_dim)
        self.w2 = np.random.normal(0, np.sqrt(2 / hidden_dim), size=(hidden_dim, hidden_dim))
        self.b2 = np.zeros(hidden_dim)
        self.w3 = np.random.normal(0, np.sqrt(2 / hidden_dim), size=(hidden_dim, 1))
        self.b3 = np.zeros(1)

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.h1 = np.maximum(0, np.dot(x, self.w1) + self.b1)
        self.h2 = np.maximum(0, np.dot(self.h1, self.w2) + self.b2)
        self.out = np.dot(self.h2, self.w3) + self.b3
        return self.out

    def train_step(self, x: np.ndarray, y: np.ndarray) -> float:
        n = len(x)
        y_pred = self.forward(x)
        loss = float(np.mean((y_pred - y) ** 2))

        # Backward gradient derivation
        d_out = 2.0 * (y_pred - y) / n
        dw3 = np.dot(self.h2.T, d_out) + self.weight_decay * self.w3
        db3 = np.sum(d_out, axis=0)

        dh2 = np.dot(d_out, self.w3.T) * (self.h2 > 0)
        dw2 = np.dot(self.h1.T, dh2) + self.weight_decay * self.w2
        db2 = np.sum(dh2, axis=0)

        dh1 = np.dot(dh2, self.w2.T) * (self.h1 > 0)
        dw1 = np.dot(x.T, dh1) + self.weight_decay * self.w1
        db1 = np.sum(dh1, axis=0)

        # SGD parameter update
        self.w3 -= self.lr * dw3
        self.b3 -= self.lr * db3
        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1

        return loss

    def evaluate(self, dataloader: MiniBatchLoader) -> float:
        losses = []
        for bx, by in dataloader:
            preds = self.forward(bx)
            losses.append(np.mean((preds - by) ** 2))
        return float(np.mean(losses))

## 3. End-to-End K-Fold Cross Validation Orchestration

We partition the 800 development samples into $K=5$ folds.
During each fold iteration:
1. We train for 40 epochs on the training split.
2. We record the validation loss at each epoch.
3. We generate **Out-Of-Fold (OOF)** predictions on the validation fold using the trained fold model.
4. We save the fold model weights.

In [ ]:
n_dev = len(full_dev_dataset)
num_folds = 5
indices = np.arange(n_dev)
np.random.default_rng(42).shuffle(indices)

fold_size = n_dev // num_folds
fold_models: list[FeedforwardRegressor] = []
oof_predictions = np.zeros((n_dev, 1))

print(f"Beginning {num_folds}-Fold Cross-Validation...")
print("-" * 55)

for fold in range(num_folds):
    val_mask = (indices >= fold * fold_size) & (indices < (fold + 1) * fold_size)
    val_indices = indices[val_mask]
    train_indices = indices[~val_mask]

    train_loader = MiniBatchLoader(
        full_dev_dataset,
        batch_size=32,
        indices=train_indices,
        shuffle=True,
        seed=fold,
    )
    val_loader = MiniBatchLoader(
        full_dev_dataset,
        batch_size=64,
        indices=val_indices,
        shuffle=False,
        seed=fold,
    )

    model = FeedforwardRegressor(in_dim=8, hidden_dim=32, lr=0.01)
    for _epoch in range(40):
        for bx, by in train_loader:
            model.train_step(bx, by)

    val_loss = model.evaluate(val_loader)
    # Store out-of-fold validation predictions
    val_preds = model.forward(full_dev_dataset.x[val_indices])
    oof_predictions[val_indices] = val_preds
    fold_models.append(model)

    print(f"Fold {fold + 1}/{num_folds} Complete | Validation MSE: {val_loss:.4f}")

# Compute overall Out-Of-Fold Cross-Validation MSE
oof_mse = float(np.mean((oof_predictions - full_dev_dataset.y) ** 2))
print("-" * 55)
print(f"Overall Out-Of-Fold (OOF) CV MSE: {oof_mse:.4f}")

## 4. Model Ensembling & Holdout Test Verification

In competitive machine learning, the single best predictor is rarely a single model; it is an **ensemble blend** formed by averaging predictions across all $K$ fold models:

$$\hat{y}_{\text{ensemble}}(x) = \frac{1}{K} \sum_{k=1}^K \hat{f}_{\text{fold } k}(x)$$

We evaluate each individual fold model on the unseen holdout test set (`X_test`), and contrast them against the ensemble average.

In [ ]:
test_loader = MiniBatchLoader(holdout_test_dataset, batch_size=64, shuffle=False)

fold_test_mses = []
test_preds_all_folds = np.zeros((num_folds, len(holdout_test_dataset)))

for k, model in enumerate(fold_models):
    t_loss = model.evaluate(test_loader)
    fold_test_mses.append(t_loss)
    test_preds_all_folds[k, :] = model.forward(holdout_test_dataset.x).ravel()
    print(f"Fold {k + 1} Holdout Test MSE: {t_loss:.4f}")

# Compute ensemble predictions (simple mean across all K models)
ensemble_test_preds = np.mean(test_preds_all_folds, axis=0).reshape(-1, 1)
ensemble_test_mse = float(np.mean((ensemble_test_preds - holdout_test_dataset.y) ** 2))

print("=" * 55)
print(f"Average Single Fold Test MSE: {np.mean(fold_test_mses):.4f}")
print(f"Ensemble Test MSE:            {ensemble_test_mse:.4f}")
improvement = (np.mean(fold_test_mses) - ensemble_test_mse) / np.mean(fold_test_mses)
print(f"Ensemble Variance Reduction:  {improvement * 100:.2f}% improvement")

## Summary: Validation & Deployment Blueprint

1. **Information Isolation:** Feature transformations (scaling, imputation, target encoding) must be fitted strictly inside training folds. Fitting scalers on the full dataset leaks validation variance into the training loop.
2. **OOF Alignment:** When the Out-Of-Fold CV score improves, the private holdout test score should improve in tandem. If CV improves but test degrades, inspect the pipeline for data leakage or covariate shift.
3. **Ensemble Benefit:** Ensembling models from $K$-fold cross-validation directly reduces parameter variance without requiring retraining on the combined dataset.